In [1]:
!pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cu132
!apt-get -y install aria2

Looking in indexes: https://download.pytorch.org/whl/cu132
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
aria2 is already the newest version (1.37.0+debian-1build3).
0 upgraded, 0 newly installed, 0 to remove and 53 not upgraded.


In [2]:
!git clone https://github.com/ussoewwin/ComfyUI-SeedVR2_VideoUpscaler.git

fatal: destination path 'ComfyUI-SeedVR2_VideoUpscaler' already exists and is not an empty directory.


In [3]:
%cd ComfyUI-SeedVR2_VideoUpscaler

/ComfyUI-SeedVR2_VideoUpscaler


In [4]:
!aria2c --console-log-level=error --allow-overwrite=true -x 16 -s 16 -k 1M "https://huggingface.co/Comfy-Org/SeedVR2/resolve/main/vae/seedvr2_ema_vae_fp16.safetensors" -d . -o seedvr2_ema_vae_fp16.safetensors

[#54a91a 368MiB/478MiB(77%) CN:16 DL:259MiB][35m]
Download Results:
gid   |stat|avg speed  |path/URI
======+====+===========+=======================================================
54a91a|OK  |   277MiB/s|./seedvr2_ema_vae_fp16.safetensors

Status Legend:
(OK):download completed.


In [4]:
!pip install -r requirements.txt

In [5]:
!pip install tensorrt-rtx==1.6.1.120 onnx onnxscript polygraphy

In [6]:
!mkdir -p models/SEEDVR2 && mv seedvr2_ema_vae_fp16.safetensors models/SEEDVR2/

mv: cannot stat 'seedvr2_ema_vae_fp16.safetensors': No such file or directory


In [7]:
!git pull

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 1.54 KiB | 527.00 KiB/s, done.
From https://github.com/ussoewwin/ComfyUI-SeedVR2_VideoUpscaler
   aee056d..90f9817  main       -> origin/main
Updating aee056d..90f9817
Fast-forward
 tools/cloud_build_engine.py | 70 +++++++++++++++++++++++++++++++++++++++++----
 1 file changed, 64 insertions(+), 6 deletions(-)


In [8]:
!python3 tools/cloud_export_gpu.py --repo . --kind encoder --frames 85 --tile 512 --output vae_encoder_85f_tile512.onnx --model seedvr2_ema_vae_fp16.safetensors
!python3 tools/cloud_build_engine.py vae_encoder_85f_tile512.onnx --output vae_encoder_85f_tile512.rtxplan --workspace-gb 16

GPU: NVIDIA GeForce RTX 5090
VRAM: 31.4 GiB, arch: sm_120
📊 Initial CUDA memory: 30.90GB free / 31.40GB total
⚠️  SeedVR2 optimizations check: SageAttention ❌ | Flash Attention ❌ | Triton ✅
💡 Optional: pip install sageattention flash-attn
🔧 Conv3d workaround active: PyTorch 2.14.0, cuDNN 92400 (fixing VAE 3x memory bug)
Instantiating VideoAutoencoderKL on meta device...
Loading VAE weights from models/SEEDVR2/seedvr2_ema_vae_fp16.safetensors to CUDA...
There are modules in VideoAutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.
Exporting 85f encoder ONNX on GPU (legacy exporter; no chunking)...
/ComfyUI-SeedVR2_VideoUpscaler/src/models/video_vae_v3/modules/causal_inflation_lib.py:380: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will 

In [ ]:
!python3 tools/cloud_export_gpu.py --repo . --kind encoder --frames 53 --tile 256 --output vae_encoder_57f_tile256.onnx --model seedvr2_ema_vae_fp16.safetensors
!python3 tools/cloud_build_engine.py vae_encoder_57f_tile256.onnx --output vae_encoder_57f_tile256.rtxplan --workspace-gb 8

In [59]:
!python3 tools/cloud_export_gpu.py --repo . --kind decoder --frames 21 --tile 256 --output vae_decoder_tile_256_21f.onnx --model seedvr2_ema_vae_fp16.safetensors
!python3 tools/cloud_build_engine.py vae_decoder_tile_256_21f.onnx --output vae_decoder_tile_256_21f.rtxplan --workspace-gb 8

GPU: NVIDIA GeForce RTX 5090
VRAM: 31.3 GiB, arch: sm_120
📊 Initial CUDA memory: 30.85GB free / 31.35GB total
⚠️  SeedVR2 optimizations check: SageAttention ❌ | Flash Attention ❌ | Triton ✅
💡 Optional: pip install sageattention flash-attn
🔧 Conv3d workaround active: PyTorch 2.13.0, cuDNN 92000 (fixing VAE 3x memory bug)
Instantiating VideoAutoencoderKL on meta device...
Loading VAE weights from models/SEEDVR2/seedvr2_ema_vae_fp16.safetensors to CUDA...
There are modules in VideoAutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.
Exporting 21f decoder ONNX on GPU (legacy tracer)...
/ComfyUI-SeedVR2_VideoUpscaler/src/models/video_vae_v3/modules/causal_inflation_lib.py:380: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a

In [ ]:
!git pull